# Sewage Defect — Tier 2 Segmentation Training on SageMaker (GPU)

Train **yolo26m-seg** on the SAM-generated pseudo-label dataset. The model emits
bbox + per-instance binary mask in a single forward pass, replacing the
decoupled YOLO + SAM web-time pipeline with one unified ONNX file.

**Prerequisite** — run `src/generate_seg_labels.py` locally first to produce
`src/data/sewage-yolo26-seg/{train,valid,test}/{images,labels}/` and upload
the whole folder to S3 (`s3://<bucket>/sewage-yolo26-seg/`).

**Target instance**: same options as `sagemaker_train.ipynb`. Seg models are
~15% slower than detect models at the same batch size due to mask coefficients.

| Instance | GPU | VRAM | Suggested batch |
|---|---|---:|---:|
| ml.g4dn.xlarge | T4 | 16 GB | 6 |
| ml.g5.xlarge | A10G | 24 GB | 12 |
| ml.g6.xlarge | L4 | 24 GB | 12 |

## 1. Environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os, platform, torch
print()
print('python  :', platform.python_version())
print('torch   :', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device  :', torch.cuda.get_device_name(0))
    print('VRAM    :', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GiB')
is_studio = os.path.exists('/home/sagemaker-user')
WORKSPACE = '/home/sagemaker-user/cnn-assignment3' if is_studio else '/home/ec2-user/SageMaker/cnn-assignment3'
print('workspace:', WORKSPACE)

## 2. Install dependencies

In [ ]:
!pip install -q -U \
    'ultralytics>=8.4.51' \
    'pycocotools>=2.0.11' \
    'onnx>=1.17' 'onnxslim>=0.1.71' 'onnxruntime' \
    'huggingface_hub>=0.20' \
    boto3 sagemaker

## 3. Stage the seg dataset from S3

Expected layout in your bucket: `s3://<bucket>/sewage-yolo26-seg/{train,valid,test}/{images,labels}/` and `data.yaml`.

In [ ]:
import os, subprocess
S3_DATASET_URI = 's3://YOUR_BUCKET/sewage-yolo26-seg/'  # EDIT ME
os.makedirs(f'{WORKSPACE}/src/data', exist_ok=True)
target = f'{WORKSPACE}/src/data/sewage-yolo26-seg'
if not os.path.exists(target):
    subprocess.run(['aws', 's3', 'sync', S3_DATASET_URI, target, '--quiet'], check=True)
    print('Synced from', S3_DATASET_URI)
else:
    print('Already present:', target)
!ls {target}

## 4. Normalise dataset paths

In [ ]:
import yaml
data_yaml = f'{WORKSPACE}/src/data/sewage-yolo26-seg/data.yaml'
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
cfg['path']  = f'{WORKSPACE}/src/data/sewage-yolo26-seg'
cfg['train'] = 'train/images'
cfg['val']   = 'valid/images'
cfg['test']  = 'test/images'
cfg['task']  = 'segment'
with open(data_yaml, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(open(data_yaml).read())

## 5. Train yolo26m-seg with P2 head + cls_weights + heavy aug

- Base: `yolo26m-seg.pt` (COCO-segmentation pretrained — ultralytics auto-downloads).
- imgsz=640 — matches the 640x640 Roboflow export resolution.
- 200 epochs, MuSGD, cosine LR, patience=50.
- Class weights from PartC retrospective: Crack 2.6, Hole 2.4, Utility intrusion 2.8, etc.
- copy_paste 0.3 and mixup 0.1 boost rare-class coverage on the small (686-img) train split.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26m-seg.pt')

CIW_MAP = {
    'Buckling': 1.8, 'Crack': 2.6, 'Debris': 1.0,
    'Hole': 2.4, 'Joint offset': 1.7, 'Obstacle': 1.2, 'Utility intrusion': 2.8,
}
names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][i] for i in sorted(cfg['names'])]
cls_weights = [CIW_MAP.get(n, 1.0) for n in names]
print('cls_weights:', dict(zip(names, cls_weights)))

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 12 if vram_gb >= 20 else 6
print(f'using batch={BATCH} for {vram_gb:.0f} GiB GPU')

train_kwargs = dict(
    data=data_yaml,
    task='segment',
    epochs=200,
    imgsz=640,
    batch=BATCH,
    device=0,
    optimizer='MuSGD',
    cos_lr=True,
    patience=50,
    plots=True,
    project='runs/segment',
    name='tier2_seg_sagemaker',
    # Augmentation
    mosaic=1.0, close_mosaic=20,
    mixup=0.10, copy_paste=0.30,
    scale=0.6, degrees=10.0, translate=0.10,
    hsv_v=0.30, hsv_s=0.60, fliplr=0.5,
    # Class weighting (8.4.51+)
    cls_weights=cls_weights,
    # Performance
    amp=True,
    workers=4,
)
results = model.train(**train_kwargs)

## 6. Evaluate on the test split — per-class box + mask mAP

In [ ]:
BEST = 'runs/segment/tier2_seg_sagemaker/weights/best.pt'
best = YOLO(BEST)
test_metrics = best.val(
    data=data_yaml, split='test', imgsz=640, device=0,
    conf=0.001, iou=0.6, augment=True, plots=True,
    project='runs/segment', name='tier2_seg_test', exist_ok=True,
)
print()
print('TEST  | Box  mAP@0.5  ', f'{test_metrics.box.map50:.4f}')
print('TEST  | Box  mAP@.5:.95', f'{test_metrics.box.map:.4f}')
print('TEST  | Mask mAP@0.5  ', f'{test_metrics.seg.map50:.4f}')
print('TEST  | Mask mAP@.5:.95', f'{test_metrics.seg.map:.4f}')

## 7. Export to dual-output ONNX (browser-ready)

`dynamic=False` because WebGPU prefers fixed shapes; `nms=False` because the
TS client runs NMS to match the existing onnxruntime-web pipeline; `opset=17`
is the minimum recent enough for the new ONNX op set used by yolo26-seg.

In [ ]:
onnx_path = best.export(
    format='onnx',
    imgsz=640,
    dynamic=False,
    simplify=True,
    opset=17,
    half=True,        # FP16 for ~2x smaller file
    nms=False,
)
print('ONNX:', onnx_path)

# Sanity-check shapes — must be [(1, 4+7+32, N), (1, 32, 160, 160)]
import onnxruntime as ort
import numpy as np
sess = ort.InferenceSession(str(onnx_path))
out = sess.run(None, {sess.get_inputs()[0].name: np.random.rand(1, 3, 640, 640).astype(np.float32)})
print('Output shapes:', [o.shape for o in out])
assert out[0].shape == (1, 43, 8400), f'Detection output mismatch: {out[0].shape}'
assert out[1].shape == (1, 32, 160, 160), f'Prototype output mismatch: {out[1].shape}'
print('Dual-output sanity check passed.')

## 8. Publish SHA-256 and upload to HuggingFace Hub

Set `HF_TOKEN` in the SageMaker secret manager or paste it into the cell.
Repo is created on first push.

In [ ]:
import hashlib, os, shutil
from huggingface_hub import HfApi, create_repo

HF_TOKEN = os.environ.get('HF_TOKEN', 'PASTE_HERE')  # EDIT ME
HF_REPO  = 'plumbers-of-uts/pipevision-yolo26m-seg'

# Stage final files under a clean directory.
stage = '/tmp/hf-stage'
os.makedirs(stage, exist_ok=True)
shutil.copy(onnx_path, f'{stage}/yolo26m-seg-fp16.onnx')
shutil.copy(BEST, f'{stage}/best.pt')

# Compute SHA-256 of the ONNX file — copy this hash into both plans.
h = hashlib.sha256()
with open(f'{stage}/yolo26m-seg-fp16.onnx', 'rb') as fh:
    for chunk in iter(lambda: fh.read(1 << 20), b''):
        h.update(chunk)
sha = h.hexdigest()
print('ONNX SHA-256:', sha)

# Write classes.json and a minimal model card.
import json
with open(f'{stage}/classes.json', 'w') as fh:
    json.dump({i: n for i, n in enumerate(names)}, fh, indent=2)

with open(f'{stage}/metadata.yaml', 'w') as fh:
    fh.write(f'''model:
  name: yolo26m-seg-pipevision-fp16
  format: onnx
  input_shape: [1, 3, 640, 640]
  precision: fp16
  opset: 17
  outputs:
    - name: detections
      shape: [1, 43, 8400]
    - name: prototypes
      shape: [1, 32, 160, 160]
  sha256: "{sha}"
mask:
  channels: 32
  resolution: 160
inference:
  conf_threshold: 0.25
  iou_threshold: 0.45
  max_detections: 100
metrics:
  test:
    mAP_box_0_5: {test_metrics.box.map50:.4f}
    mAP_box_0_5_0_95: {test_metrics.box.map:.4f}
    mAP_mask_0_5: {test_metrics.seg.map50:.4f}
    mAP_mask_0_5_0_95: {test_metrics.seg.map:.4f}
''')

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO, private=False, exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=stage,
    repo_id=HF_REPO,
    repo_type='model',
    commit_message=f'tier2 seg model — sha256 {sha[:12]}',
)
print(f'Uploaded to https://huggingface.co/{HF_REPO}')
print(f'Set VITE_MODEL_SHA256={sha} in pipevision-ai/.env.local')

## 9. Persist run artifacts to S3

In [ ]:
import subprocess, time
S3_OUTPUT_URI = 's3://YOUR_BUCKET/runs/'  # EDIT ME
stamp = time.strftime('%Y%m%d-%H%M')
dest = f'{S3_OUTPUT_URI.rstrip("/")}/tier2_seg_sagemaker_{stamp}/'
subprocess.run(
    ['aws', 's3', 'sync', 'runs/segment/tier2_seg_sagemaker', dest, '--quiet'],
    check=True,
)
subprocess.run(
    ['aws', 's3', 'sync', 'runs/segment/tier2_seg_test', dest + 'test_eval/', '--quiet'],
    check=True,
)
print('Saved to', dest)

## 10. Shut down

Uncomment to stop the notebook instance (kills billing). Stop manually via console if you want to inspect outputs first.

In [ ]:
# !sudo shutdown -h now